## Import libraries

In [27]:
import pandas as pd
import numpy as np

## Load Clean Data

In [28]:
df = pd.read_csv("../data/processed/cleaned_complaints.csv")

df.head()


,complaint_id,citizen_id,category,sub_category,ward,reported_date,status,people_affected,vulnerable_population,essential_service,public_safety_risk,recurrence_indicator,estimated_effort_hours,severity
0,C000001,P0869,Drainage,Drain Blockage,Peenya,2025-10-01,Closed,230.0,39.0,No,Medium,No,4.0,High
1,C000002,P0993,Sewerage,Sewage Overflow,Hennur,2025-10-01,Resolved,70.0,14.0,Yes,Critical,No,8.0,Medium
2,C000003,P1709,Street Infrastructure,Fallen Tree Branch,Dasarahalli,2025-10-01,Closed,190.0,12.0,No,Medium,No,12.0,Medium
3,C000004,P1458,Revenue Department,Trade Licence Issue,Chickpet,2025-10-01,Resolved,7.0,2.0,No,Low,No,2.0,Low
4,C000005,P1565,Water Supply,No Water Supply,Vasanthnagar,2025-10-01,Closed,140.0,21.0,Yes,Low,No,12.0,High


## Check data

In [29]:
df.shape

(8953, 14)

In [30]:
df.columns

Index(['complaint_id', 'citizen_id', 'category', 'sub_category', 'ward',
       'reported_date', 'status', 'people_affected', 'vulnerable_population',
       'essential_service', 'public_safety_risk', 'recurrence_indicator',
       'estimated_effort_hours', 'severity'],
      dtype='str')

In [31]:
df.count()


complaint_id              8953
citizen_id                8953
category                  8953
sub_category              8953
ward                      8953
reported_date             8953
status                    8953
people_affected           8953
vulnerable_population     8953
essential_service         8834
public_safety_risk        8953
recurrence_indicator      8953
estimated_effort_hours    8737
severity                  8953
dtype: int64

## Group complaints into incidents

In [32]:
grouping_columns = [
    "category",
    "sub_category",
    "ward",
    "status"
]

df_incidents = (
    df.groupby(grouping_columns)
      .agg(
          complaint_count=("complaint_id", "count"),
          unique_citizen_count=("citizen_id", "nunique")
      )
      .reset_index()
)

df_incidents.head()

,category,sub_category,ward,status,complaint_count,unique_citizen_count
0,Drainage,Damaged Drain,Arekere,Resolved,1,1
1,Drainage,Damaged Drain,BTM Layout,Closed,1,1
2,Drainage,Damaged Drain,Basavanagudi,In Progress,1,1
3,Drainage,Damaged Drain,Begur,Closed,3,3
4,Drainage,Damaged Drain,Begur,Resolved,1,1


## Assign incident IDs

In [33]:
df_incidents.insert(
    0,
    "incident_id",
    [f"I{i:04d}" for i in range(1, len(incidents) + 1)]
)

df_incidents.head()

,incident_id,category,sub_category,ward,status,complaint_count,unique_citizen_count
0,I0001,Drainage,Damaged Drain,Arekere,Resolved,1,1
1,I0002,Drainage,Damaged Drain,BTM Layout,Closed,1,1
2,I0003,Drainage,Damaged Drain,Basavanagudi,In Progress,1,1
3,I0004,Drainage,Damaged Drain,Begur,Closed,3,3
4,I0005,Drainage,Damaged Drain,Begur,Resolved,1,1


## Rank incidents by complaint count

In [34]:
df_incidents = df_incidents.sort_values(
    by="complaint_count",
    ascending=False
).reset_index(drop=True)

df_incidents.insert(
    1,
    "priority_rank",
    range(1, len(df_incidents) + 1)
)

df_incidents.head(10)

,incident_id,priority_rank,category,sub_category,ward,status,complaint_count,unique_citizen_count
0,I2367,1,Solid Waste Management,Garbage Not Collected,Mathikere,Closed,51,37
1,I2345,2,Solid Waste Management,Garbage Not Collected,Kadugodi,Closed,49,38
2,I2370,3,Solid Waste Management,Garbage Not Collected,Mathikere,Resolved,36,28
3,I0652,4,Electrical,Street Light Not Working,Mathikere,Closed,32,30
4,I2305,5,Solid Waste Management,Garbage Not Collected,Dasarahalli,Closed,28,23
5,I2347,6,Solid Waste Management,Garbage Not Collected,Kadugodi,Resolved,27,26
6,I2333,7,Solid Waste Management,Garbage Not Collected,Hosahalli,Resolved,26,23
7,I0544,8,Electrical,Multiple Street Lights Not Working,Sadashivanagar,Resolved,26,19
8,I2393,9,Solid Waste Management,Garbage Not Collected,Shantinagar,Closed,26,21
9,I2294,10,Solid Waste Management,Garbage Not Collected,Bellandur,Resolved,25,23


## Save the output

In [36]:
df_incidents.to_csv(
    "../data/processed/incidents.csv",
    index=False
)

In [37]:
pd.read_csv("../data/processed/incidents.csv").head()

,incident_id,priority_rank,category,sub_category,ward,status,complaint_count,unique_citizen_count
0,I2367,1,Solid Waste Management,Garbage Not Collected,Mathikere,Closed,51,37
1,I2345,2,Solid Waste Management,Garbage Not Collected,Kadugodi,Closed,49,38
2,I2370,3,Solid Waste Management,Garbage Not Collected,Mathikere,Resolved,36,28
3,I0652,4,Electrical,Street Light Not Working,Mathikere,Closed,32,30
4,I2305,5,Solid Waste Management,Garbage Not Collected,Dasarahalli,Closed,28,23


In [38]:
print("Cleaned dataset shape:", df.shape)
print("Grouped dataset shape:", df_incidents.shape)

Cleaned dataset shape: (8953, 14)
Grouped dataset shape: (3053, 8)
